In [12]:

# !pip install pandas pyarrow scikit-learn seaborn matplotlib 

In [13]:
import pandas as pd
import pyarrow
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pickle



In [14]:
import mlflow

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('nyx-taxi-experiment')

<Experiment: artifact_location='/Users/dubai/MLOps_course/mlops_zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1786957730460, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786957730460, lifecycle_stage='active', name='nyx-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, Ridge


from sklearn.metrics import root_mean_squared_error

In [10]:
train_file = 'data/green_tripdata_2021-01.parquet'
val_file = 'data/green_tripdata_2021-02.parquet'
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime 
    df['duration'] = df['duration'].apply(lambda td: td.total_seconds() / 60)

    df = df[((df.duration >=1) & (df.duration <=60))]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [5]:
df_train = read_dataframe(train_file)
df_val = read_dataframe(val_file)

print(len(df_train), len(df_val))

df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

73908 61921


In [6]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [7]:
lr = LinearRegression()

X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_true=y_val, y_pred=y_pred)

7.480873453689756

In [8]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [11]:
with mlflow.start_run():
    mlflow.set_tag('developer', 'aigerim')
    mlflow.log_param('train-data-path', train_file)
    mlflow.log_param('valid-data-path', val_file)

    alpha = 0.01

    mlflow.log_param('alpha', alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric('rmse', rmse)
